# TIPS-Treasury synthetic replication (Fleckenstein, Longstaff and Lustig, 2014)

Python cross-check of `TIPS_Treasury_Replication.xlsx`. The workbook is the source of the market inputs
(Bloomberg, settlement 3/20/2026); this notebook reads them, rebuilds the Table II replication, and runs
sensitivities to the modelling choices.

Pair: **T 3.500% 2/15/2033** (target) vs **TII 1.125% 1/15/2033** (31-day maturity mismatch).

Cash flows per $100 notional, semiannual period *t*:

* inflation swap fixed leg: $F_t = (1+f_t)^{t}$, with $t$ in years (Act/365.25 from settlement)
* STRIPS notional: $x_t = c - sF_t$ for $t<T$, and $x_T = (c+100) - (s+100)F_T$
* synthetic price: $V + \sum_t x_t D(t)$, with $D(t)$ the STRIPS price per $1 face
* mispricing: $P - (V + \sum_t x_t D(t))$; positive means the Treasury is rich, negative means TIPS are rich

$I_t$ is measured from the trade date, so the TIPS position is sized per $100 of current inflation-adjusted
principal and $V$ is the real (unindexed) full price; the index ratio cancels.

In [1]:
import numpy as np
import pandas as pd
import openpyxl
from pathlib import Path
from scipy.interpolate import CubicSpline

WORKBOOK = Path("TIPS_Treasury_Replication.xlsx")
STRIPS_CURVE = Path("data/strips_ask_prices.xlsx")
pd.set_option("display.float_format", "{:.6f}".format)

## 1. Inputs from the workbook

In [2]:
ws = openpyxl.load_workbook(WORKBOOK, data_only=True)["Inputs"]

settle = pd.Timestamp("2026-03-20")
tsy_clean, tsy_accrued = ws["H5"].value, ws["H6"].value
tips_clean, tips_accrued = ws["H7"].value, ws["H8"].value
c = ws["B6"].value          # Treasury semiannual coupon per $100
s = ws["E6"].value          # TIPS semiannual coupon per $100
P = tsy_clean + tsy_accrued
V = tips_clean + tips_accrued

swap_tenors = np.array([ws[f"G{r}"].value for r in range(12, 19)], dtype=float)
swap_quotes = np.array([ws[f"I{r}"].value for r in range(12, 19)], dtype=float)   # % zero-coupon inflation swap

sched = pd.DataFrame({
    "date": pd.to_datetime([ws[f"B{r}"].value for r in range(29, 43)]),
    "strips_price": [ws[f"C{r}"].value for r in range(29, 43)],
})
sched["t"] = [ws[f"D{r}"].value for r in range(29, 43)]          # workbook year fractions
assert np.allclose(sched["t"], (sched["date"] - settle).dt.days / 365.25, atol=1e-6)   # Act/365.25
sched["grid_period"] = 0.5 * np.arange(1, len(sched) + 1)

print(f"P (Treasury full) = {P:.6f}   clean {tsy_clean:.6f} + accrued {tsy_accrued:.6f}")
print(f"V (TIPS full)     = {V:.6f}   clean {tips_clean:.6f} + accrued {tips_accrued:.6f}")
pd.DataFrame({"tenor": swap_tenors, "quote_%": swap_quotes})

P (Treasury full) = 96.928436   clean 96.609375 + accrued 0.319061
V (TIPS full)     = 97.675458   clean 97.476562 + accrued 0.198895


,tenor,quote_%
0,1.000000,2.704000
1,2.000000,2.522000
2,3.000000,2.477000
3,4.000000,2.451000
4,5.000000,2.420000
5,6.000000,2.444000
6,7.000000,2.425000


## 2. Replication engine

In [3]:
def swap_rate_halfyear_grid(periods):
    """Workbook convention: rate at the nearest half-year grid point; average of adjacent integer
    tenors at half years; flat 1Y below one year."""
    return np.interp(np.maximum(periods, 1.0), swap_tenors, swap_quotes)

def swap_rate_linear_exact(t):
    return np.interp(np.maximum(t, 1.0), swap_tenors, swap_quotes)

_spline = CubicSpline(swap_tenors, swap_quotes)
def swap_rate_cubic_exact(t):
    # paper uses a cubic spline (plus a seasonal adjustment not applied here); flat below 1Y
    return _spline(np.maximum(t, 1.0))

def replicate(f_pct, t, strips_price, P=P, V=V):
    F = (1 + f_pct / 100) ** t
    x = c - s * F
    x[-1] = (c + 100) - (s + 100) * F[-1]
    D = strips_price / 100.0
    strips_cost = float(np.sum(x * D))
    synthetic = V + strips_cost
    return dict(F=F, x=x, D=D, strips_cost=strips_cost, synthetic=synthetic, mispricing=P - synthetic)

## 3. Base case (matches the workbook) in Table II format

In [4]:
f_base = swap_rate_halfyear_grid(sched["grid_period"].values)
base = replicate(f_base, sched["t"].values, sched["strips_price"].values)

table2 = pd.DataFrame({
    "Date": sched["date"].dt.strftime("%m/%d/%Y"),
    "Treasury": [c] * 13 + [c + 100],
    "TIPS": [f"{s:.4f} I{i}" for i in range(1, 14)] + [f"{s + 100:.4f} I14"],
    "Inflation swap": [f"{s * F:.4f} - {s:.4f} I{i}" for i, F in enumerate(base["F"][:-1], 1)]
                      + [f"{(s + 100) * base['F'][-1]:.4f} - {s + 100:.4f} I14"],
    "STRIPS x_t": base["x"],
    "D(t) per $1": base["D"],
    "Total": [c] * 13 + [c + 100],
})
row0 = pd.DataFrame([{"Date": "0 (3/20/2026)", "Treasury": -P, "TIPS": f"{-V:.4f}", "Inflation swap": "0",
                      "STRIPS x_t": -base["strips_cost"], "D(t) per $1": np.nan, "Total": -base["synthetic"]}])
table2 = pd.concat([row0, table2], ignore_index=True)
table2

,Date,Treasury,TIPS,Inflation swap,STRIPS x_t,D(t) per $1,Total
0,0 (3/20/2026),-96.928436,-97.6755,0,-0.225127,NaN,-97.900584
1,08/15/2026,1.750000,0.5625 I1,0.5686 - 0.5625 I1,1.181386,0.985150,1.750000
2,02/15/2027,1.750000,0.5625 I2,0.5763 - 0.5625 I2,1.173691,0.968006,1.750000
3,08/15/2027,1.750000,0.5625 I3,0.5833 - 0.5625 I3,1.166748,0.948820,1.750000
4,02/15/2028,1.750000,0.5625 I4,0.5899 - 0.5625 I4,1.160119,0.930997,1.750000
5,08/15/2028,1.750000,0.5625 I5,0.5969 - 0.5625 I5,1.153068,0.913442,1.750000
6,02/15/2029,1.750000,0.5625 I6,0.6040 - 0.5625 I6,1.145983,0.896130,1.750000
7,08/15/2029,1.750000,0.5625 I7,0.6111 - 0.5625 I7,1.138879,0.878689,1.750000
8,02/15/2030,1.750000,0.5625 I8,0.6184 - 0.5625 I8,1.131646,0.861220,1.750000
9,08/15/2030,1.750000,0.5625 I9,0.6254 - 0.5625 I9,1.124598,0.843968,1.750000


In [5]:
summary = pd.Series({
    "Treasury full price P": P,
    "TIPS full price V": V,
    "STRIPS cost sum x_t D(t)": base["strips_cost"],
    "Synthetic price V + STRIPS": base["synthetic"],
    "Mispricing P - synthetic": base["mispricing"],
})
summary

Treasury full price P        96.928436
TIPS full price V            97.675458
STRIPS cost sum x_t D(t)      0.225127
Synthetic price V + STRIPS   97.900584
Mispricing P - synthetic     -0.972148
dtype: float64

In [6]:
# Cash-flow check: TIPS + swaps + STRIPS must reproduce the Treasury cash flows for any inflation path.
rng = np.random.default_rng(0)
I = np.cumprod(1 + rng.normal(0.0125, 0.01, len(sched)))       # arbitrary realised index path
cf_tips = s * I; cf_tips[-1] = (s + 100) * I[-1]
notional = np.full(len(sched), s); notional[-1] = s + 100
cf_swap = notional * (base["F"] - I)
replicated = cf_tips + cf_swap + base["x"]
target = np.full(len(sched), c); target[-1] = c + 100
assert np.allclose(replicated, target)
print("Replicated cash flows match the Treasury in every period.")

Replicated cash flows match the Treasury in every period.


## 4. Sensitivities

* **As submitted**: TIPS accrued interest of 0.261864 (the figure used in the report) instead of the
  64-day real accrued of 0.198895.
* **Linear, exact t**: swap rate interpolated at each cash flow's actual year fraction rather than the
  nearest half-year grid point.
* **Cubic spline, exact t**: closer to the paper's method (the paper also applies a CPI seasonal
  adjustment, not done here).
* **STRIPS from `data/strips_ask_prices.xlsx`**: the full STRIPS ask curve exported for this project.
  Its prices at several Feb/Aug coupon dates differ slightly from the prices in the workbook.

In [7]:
curve = pd.read_excel(STRIPS_CURVE)
curve["Maturity"] = pd.to_datetime(curve["Maturity"])
curve_px = curve.set_index("Maturity")["Ask Price"].reindex(sched["date"]).values
assert not np.isnan(curve_px).any()

t, px = sched["t"].values, sched["strips_price"].values
V_submitted = tips_clean + 0.261864
cases = {
    "Base (corrected workbook)": replicate(f_base, t, px),
    "As submitted (TIPS accrued 0.261864)": replicate(f_base, t, px, V=V_submitted),
    "Linear interpolation at exact t": replicate(swap_rate_linear_exact(t), t, px),
    "Cubic spline at exact t": replicate(swap_rate_cubic_exact(t), t, px),
    "STRIPS prices from strips_ask_prices.xlsx": replicate(f_base, t, curve_px),
}
pd.DataFrame({k: {"STRIPS cost": v["strips_cost"], "Synthetic": v["synthetic"], "Mispricing": v["mispricing"]}
              for k, v in cases.items()}).T

,STRIPS cost,Synthetic,Mispricing
Base (corrected workbook),0.225127,97.900584,-0.972148
As submitted (TIPS accrued 0.261864),0.225127,97.963553,-1.035117
Linear interpolation at exact t,0.214226,97.889684,-0.961248
Cubic spline at exact t,0.181004,97.856461,-0.928026
STRIPS prices from strips_ask_prices.xlsx,0.226031,97.901489,-0.973053


In [8]:
pd.DataFrame({"date": sched["date"].dt.date, "workbook STRIPS": px, "strips_ask_prices.xlsx": curve_px,
              "difference": curve_px - px})

,date,workbook STRIPS,strips_ask_prices.xlsx,difference
0,2026-08-15,98.515000,98.514993,-0.000007
1,2027-02-15,96.800600,96.798917,-0.001683
2,2027-08-15,94.882000,94.880713,-0.001287
3,2028-02-15,93.099700,93.092699,-0.007001
4,2028-08-15,91.344200,91.342046,-0.002154
5,2029-02-15,89.613000,89.607846,-0.005154
6,2029-08-15,87.868900,87.857150,-0.011750
7,2030-02-15,86.122000,86.125299,0.003299
8,2030-08-15,84.396800,84.400419,0.003619
9,2031-02-15,82.673100,82.681041,0.007941


## 5. Reading the result

The mispricing is about **-$0.97 per $100** in every variant (range about -0.93 to -1.04): the replicating
portfolio costs more than the Treasury, so TIPS are rich relative to nominal Treasuries on 3/20/2026. That is the
opposite sign to FLL's 2004-2009 sample, where Treasuries were rich, and two orders of magnitude smaller than the
$23.10 in their December 2008 example. The result is not adjusted for the 31-day maturity mismatch, for the TIPS
paying on Jan/Jul 15 instead of Feb/Aug 15, for CPI seasonality, or for bid-ask costs, which the paper puts at
roughly 20-70 cents per $100 depending on maturity, so it is within the range where it may not be a tradable
arbitrage.